<!-- @format -->

# Adult Income Prediction

## 2. Data Pre-processing

Sau bước EDA, nhóm tiến hành tiền xử lý dữ liệu để chuẩn bị cho giai đoạn huấn luyện mô hình.  
Các bước chính gồm:

- chuẩn hóa missing value,
- loại bỏ đặc trưng dư thừa,
- chuẩn bị tập đặc trưng và biến mục tiêu,
- chia train/test,
- xây dựng và so sánh các cấu hình preprocessing cho numerical và categorical features.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import modules.preprocessing as prep

url = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"
df = pd.read_csv(url)
print("Dataset shape:", df.shape)

Dataset shape: (48842, 15)


<!-- @format -->

### 2.1. Missing value standardization

Kết quả cho thấy dữ liệu thiếu không xuất hiện dưới dạng `NaN` ngay từ đầu mà được mã hóa bằng ký hiệu `?`.  
Sau khi thay thế, các cột có missing value là:

- `workclass`
- `occupation`
- `native-country`

Tỷ lệ thiếu ở các cột này không quá lớn, vì vậy nhóm ưu tiên giữ lại dữ liệu và sẽ xử lý bằng imputation trong pipeline thay vì loại bỏ toàn bộ các dòng.


In [2]:
# [Preprocessing 2.1] Chuẩn hóa '?' về NaN và kiểm tra missing value
df_prep = df.copy()
df_prep.replace('?', np.nan, inplace=True)

print(df_prep.isnull().sum())

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64


<!-- @format -->

### 2.2. Remove redundant and non-predictive features

Dựa trên kết quả EDA, nhóm loại bỏ hai cột sau:

- `education`: do trùng lặp thông tin với `educational-num`
- `fnlwgt`: đây là sampling weight của điều tra dân số, không phải đặc trưng mô tả trực tiếp cá nhân và thường không mang nhiều ý nghĩa dự đoán trong bài toán phân loại thu nhập

Việc loại bỏ hai cột này giúp giảm dư thừa thông tin và làm tập đặc trưng phù hợp hơn cho mô hình.


In [3]:
# [Preprocessing 2.2] Loại bỏ các cột dư thừa / không có ý nghĩa dự đoán
df_prep = prep.drop_columns(df_prep, columns=['education', 'fnlwgt'])

print(df_prep.shape)
print(df_prep.columns.tolist())
df_prep.head()

Dropped columns: ['education', 'fnlwgt']
(48842, 13)
['age', 'workclass', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


,age,workclass,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


<!-- @format -->

### 2.3. Split features and target

Sau khi tách dữ liệu:

- tập đặc trưng `X` gồm 12 biến,
- biến mục tiêu `y` là `income`.

Phân phối của biến mục tiêu cho thấy số mẫu thuộc lớp `<=50K` lớn hơn đáng kể so với lớp `>50K`, nghĩa là bài toán có hiện tượng mất cân bằng lớp nhẹ.  
Vì vậy, ở bước chia train/test, nhóm sẽ dùng `stratify=y` để giữ phân phối lớp tương đối ổn định giữa hai tập.


In [4]:
# [Preprocessing 2.3] Tách features và target
X = df_prep.drop(columns=['income'])
y = df_prep['income']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (48842, 12)
y shape: (48842,)

Target distribution:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


<!-- @format -->

### 2.4. Identify numerical and categorical features

Các biến được chia thành hai nhóm để phục vụ bước tiền xử lý:

- **Numerical features**: `age`, `educational-num`, `capital-gain`, `capital-loss`, `hours-per-week`
- **Categorical features**: `workclass`, `marital-status`, `occupation`, `relationship`, `race`, `gender`, `native-country`

Việc tách riêng hai nhóm này giúp xây dựng pipeline phù hợp:

- biến số sẽ được xử lý theo các cấu hình preprocessing khác nhau,
- biến phân loại sẽ được xử lý missing value và encoding trước khi đưa vào mô hình.


In [5]:
# [Preprocessing 2.4] Xác định numerical và categorical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical features :", num_features)
print("Categorical features:", cat_features)

Numerical features : ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']


<!-- @format -->

### 2.5. Train-test split

Dữ liệu được chia theo tỷ lệ **80/20** cho train và test.  
Việc sử dụng `stratify=y` giúp giữ phân phối lớp của biến mục tiêu gần như giống nhau giữa hai tập.

Điều này đặc biệt quan trọng với bài toán phân loại có mất cân bằng lớp, vì giúp việc đánh giá mô hình ở tập test đáng tin cậy hơn.


In [6]:
# [Preprocessing 2.5] Chia train/test (80/20, stratify theo target)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

X_train shape: (39073, 12)
X_test shape : (9769, 12)

Train target distribution:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64

Test target distribution:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64


<!-- @format -->

### 2.6. Define preprocessing configurations

Để việc tiền xử lý có ý nghĩa hơn, nhóm xây dựng nhiều cấu hình preprocessing khác nhau thay vì chỉ dùng một pipeline cố định.

Các cấu hình được thay đổi theo ba yếu tố chính:

- cách xử lý missing value ở biến categorical,
- cách scaling biến numerical,
- và giữ nguyên One-Hot Encoding cho các biến categorical không có thứ tự.

Lưu ý: các biến numerical trong bộ dữ liệu này không có missing value, nên không cần áp dụng numerical imputation ở bước so sánh này.


In [7]:
# [Preprocessing 2.6] Khởi tạo toàn bộ cấu hình preprocessing từ module
preprocessing_configs = prep.get_all_classical_configs(num_features, cat_features)

print("Available preprocessing configurations:")
for name in preprocessing_configs:
    print("-", name)

Available preprocessing configurations:
- config_1_onehot_mostfreq_standard
- config_2_onehot_constant_standard
- config_3_onehot_mostfreq_minmax
- config_4_onehot_mostfreq_noscale


<!-- @format -->

### 2.7. Compare preprocessing configurations

Kết quả cho thấy các cấu hình preprocessing tạo ra số chiều đầu ra khác nhau:

- các cấu hình dùng **One-Hot Encoding** làm tăng số lượng đặc trưng,
- cấu hình dùng giá trị hằng `"Missing"` cho categorical missing có thể làm tăng thêm số category sau encoding,
- các lựa chọn scaling không làm thay đổi số chiều nhưng có thể ảnh hưởng đến hiệu năng mô hình ở bước sau.

Điều này cho thấy lựa chọn preprocessing có ảnh hưởng trực tiếp đến cách biểu diễn dữ liệu đầu vào cho mô hình.


In [8]:
# [Preprocessing 2.7] So sánh số chiều đầu ra của các cấu hình
shape_rows = []

for name, transformer in preprocessing_configs.items():
    X_tr = transformer.fit_transform(X_train)
    X_te = transformer.transform(X_test)
    shape_rows.append({
        'configuration': name,
        'X_train_shape': X_tr.shape,
        'X_test_shape' : X_te.shape,
        'n_features'   : X_tr.shape[1],
    })
    print(f"{name}")
    print(f"  X_train: {X_tr.shape}  |  X_test: {X_te.shape}")

shape_df = (
    pd.DataFrame(shape_rows)
    .sort_values(by='n_features', ascending=False)
    .reset_index(drop=True)
)
print()
print(shape_df)

config_1_onehot_mostfreq_standard
  X_train: (39073, 88)  |  X_test: (9769, 88)
config_2_onehot_constant_standard
  X_train: (39073, 91)  |  X_test: (9769, 91)
config_3_onehot_mostfreq_minmax
  X_train: (39073, 88)  |  X_test: (9769, 88)
config_4_onehot_mostfreq_noscale
  X_train: (39073, 88)  |  X_test: (9769, 88)

                       configuration X_train_shape X_test_shape  n_features
0  config_2_onehot_constant_standard   (39073, 91)   (9769, 91)          91
1  config_1_onehot_mostfreq_standard   (39073, 88)   (9769, 88)          88
2    config_3_onehot_mostfreq_minmax   (39073, 88)   (9769, 88)          88
3   config_4_onehot_mostfreq_noscale   (39073, 88)   (9769, 88)          88


<!-- @format -->

### 2.8. Evaluate preprocessing configurations with Logistic Regression

Để chọn cấu hình preprocessing phù hợp, nhóm sử dụng Logistic Regression như một mô hình đánh giá cố định và so sánh hiệu năng của các cấu hình theo các chỉ số classification.


In [9]:
# [Preprocessing 2.8] So sánh các cấu hình bằng Logistic Regression
# Hàm compare_preprocessing_configs fit lại từng transformer trên X_train
# để tránh leakage từ bước so sánh shape ở trên.
lr_results_df = prep.compare_preprocessing_configs(
    preprocessing_configs,
    X_train, X_test,
    y_train, y_test,
    pos_label='>50K'
)

print(lr_results_df)

                       configuration  accuracy  precision    recall  f1_score  \
0  config_2_onehot_constant_standard  0.855359   0.746142  0.599658  0.664928   
1   config_4_onehot_mostfreq_noscale  0.852493   0.738691  0.593670  0.658288   
2  config_1_onehot_mostfreq_standard  0.852185   0.737513  0.593670  0.657820   
3    config_3_onehot_mostfreq_minmax  0.852595   0.744022  0.585543  0.655337   

   n_features_after_preprocessing  
0                              91  
1                              88  
2                              88  
3                              88  


<!-- @format -->

### 2.9. Select best preprocessing configuration

Dựa trên kết quả so sánh với Logistic Regression, nhóm chọn cấu hình preprocessing tốt nhất để sử dụng cho các bước tiếp theo.


In [10]:
# [Preprocessing 2.9] Chọn cấu hình tốt nhất theo F1-score
best_name, best_preprocessor, X_train_best, X_test_best = prep.select_best_preprocessor(
    preprocessing_configs,
    lr_results_df,
    X_train, X_test
)

Best config selected : config_2_onehot_constant_standard
X_train_best shape   : (39073, 91)
X_test_best shape    : (9769, 91)


<!-- @format -->

Dựa trên kết quả so sánh với Logistic Regression, nhóm chọn `config_2_onehot_constant_standard` làm cấu hình preprocessing tốt nhất cho các bước tiếp theo vì đạt F1-score cao nhất trong các cấu hình đã thử.


<!-- @format -->

### Preprocessing Summary

Sau bước tiền xử lý, nhóm đã:

- Thay thế `?` bằng `NaN` để chuẩn hóa missing value
- Loại bỏ `education` (trùng với `educational-num`) và `fnlwgt` (sampling weight)
- Chia train/test theo tỷ lệ 80/20 với `stratify=y`
- Thử nghiệm 4 cấu hình preprocessing và chọn `config_2_onehot_constant_standard` dựa trên F1-score

Dữ liệu sau preprocessing có **91 đặc trưng**, sẵn sàng cho bước huấn luyện mô hình.
